In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import pandas as pd

df = pd.read_csv(f"{path}/Q1_data.csv")

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
import matplotlib.pyplot as plt

plt.hist(df["Delivery_Time"], bins=30)
plt.xlabel("Delivery Time (minutes)")
plt.ylabel("Frequency")
plt.title("Distribution of Delivery Time")
plt.show()


In [ ]:
# Task 1: Write your code here:
df = df.drop(columns=["Order_ID"])


In [ ]:
# Task 2: Write your code here:
for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].fillna(df[col].mode()[0])
    else:
        df[col] = df[col].fillna(df[col].median())


In [ ]:
# Task 3: Write your code here:
df = df.drop_duplicates()


In [ ]:
# Task 4: Write your code here:
df = pd.get_dummies(df, drop_first=True)


In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

X = df.drop(columns=["Delivery_Time"])
y = df["Delivery_Time"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [ ]:
# Task 1: Write your code here:
import numpy as np
from sklearn.model_selection import KFold, cross_val_score
from sklearn.ensemble import RandomForestRegressor

X = df.drop(columns=["Delivery_Time"])
y = df["Delivery_Time"]


In [ ]:
# Task 2,3,4,5: Write your code here:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

model = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

mae_scores = -cross_val_score(model, X, y, cv=kf, scoring="neg_mean_absolute_error")

print("MAE per fold:", mae_scores)
print("Average MAE:", mae_scores.mean())


In [ ]:
# Task 1: Write your code here:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

X = df.drop(columns=["Delivery_Time"])
y = df["Delivery_Time"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

importances = model.feature_importances_
idx = np.argsort(importances)[-15:]  # top 15

plt.barh(X.columns[idx], importances[idx])
plt.xlabel("Importance")
plt.title("Top Feature Importances (RandomForest)")
plt.show()


In [ ]:
# Task 2: Write your code here:
y_pred = model.predict(X_test)

plt.hist(y_pred, bins=30)
plt.xlabel("Predicted Delivery Time (minutes)")
plt.ylabel("Frequency")
plt.title("Histogram of Predicted Delivery Time")
plt.show()


In [ ]:
!pip -q install catboost


In [ ]:
# Task Bonus: Write your code here:
import numpy as np
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from catboost import CatBoostRegressor
X = df.drop(columns=["Delivery_Time"]).values
y = df["Delivery_Time"].values

kf = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []

for train_idx, val_idx in kf.split(X):
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    rf = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
    cb = CatBoostRegressor(iterations=500, learning_rate=0.05, depth=8, verbose=0, random_state=42)

    rf.fit(X_train, y_train)
    cb.fit(X_train, y_train)

    pred_rf = rf.predict(X_val)
    pred_cb = cb.predict(X_val)

    pred_avg = (pred_rf + pred_cb) / 2.0

    mae_scores.append(mean_absolute_error(y_val, pred_avg))

print("MAE per fold:", mae_scores)
print("Average MAE:", np.mean(mae_scores))
